In [1]:
import aspose.words_foss as aw 

docs_path = './docs/'
filename = "uk-rf.doc"

doc = aw.Document(docs_path+filename)
text = doc.get_text()

In [2]:
import numpy as np

def split_into_chunks(text: str, chunk_size: int=800,  overlap: int=80) -> list[str]:
    # для каждого чанка выведем начальный и конечный индексы, чтобы 
    if overlap >= chunk_size:
        raise ValueError("chunk_size должен быть больше overlap!")

    if overlap < 0:
        raise ValueError("overlap не должен быть меньше нуля!")

    if chunk_size < 0:
        raise ValueError("chunk_size не может быть меньше нуля!")

    if not text:
        raise ValueError("Текст пустой!")

    start_indexes = np.arange(0, len(text), chunk_size-overlap)
    end_indexes = start_indexes + chunk_size

    chunks = [
        text[start:end] for start, end in zip(start_indexes, end_indexes)
    ]

    return chunks

In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=60)
chunks = text_splitter.split_text(text)

In [4]:
import torch
import torch.nn.functional as F
import polars as pl
import hashlib
from transformers import AutoTokenizer, AutoModel

pl.Config.set_fmt_str_lengths(1000)
torch.set_grad_enabled(False)
device = "cuda" if torch.cuda.is_available() else "cpu"

cache_dir = './models/'
embeding_model_name = "google/embeddinggemma-2"

vdb_path = './database/'
vdb_filename = 'test.parquet'

tokenizer = AutoTokenizer.from_pretrained(embeding_model_name, cache_dir=cache_dir)
model = AutoModel.from_pretrained(embeding_model_name, cache_dir=cache_dir).to(device).eval()

def get_ids(chunks, tokenizer):
    tokens = [tokenizer.tokenize(chunk) for chunk in chunks]

    ids = [tokenizer.convert_tokens_to_ids(token) for token in tokens]

    prepared_ids = tokenizer.pad(
        {"input_ids": ids},
        padding=True,
        return_attention_mask=True,
        return_tensors="pt"
    )

    return prepared_ids

def get_embeddings(batch, model):
    with torch.inference_mode():
        output = model(**batch)

    token_embeddings = output.last_hidden_state
    token_embeddings_prepared = (
        batch['attention_mask']
        .to(device)
        .to(torch.float64)
        .unsqueeze(-1)
    ) * (
        token_embeddings
    )

    return token_embeddings_prepared.mean(dim=1).cpu()

# Перешёл на langchain
chunks = text_splitter.split_text(text)

ids = tokenizer(
    chunks,
    return_tensors="pt",
    padding=True
)

batch_size = 128
tensor_len = ids['input_ids'].shape[0]

embeddings = []

for i in np.arange(0, tensor_len, batch_size):
    batch = {
        key: value[i:i+batch_size].to(device)
        for key, value in ids.items()
    }

    with torch.inference_mode():
        output = model(**batch).last_hidden_state

    embeddings.append(output.to("cpu"))

embeddings = torch.concat(embeddings, dim=0)
embeddings = (embeddings * ids['attention_mask'].unsqueeze(-1)).sum(1) / ids['attention_mask'].sum(1, keepdim=True)
embeddings = F.normalize(embeddings, dim=1)

table = pl.DataFrame({
    "id": [hashlib.sha256(chunk.encode('utf-8')).hexdigest() for chunk in chunks],
    "text": chunks,
    "embedding_vector": embeddings.to(torch.float64).numpy(),
    "doc": filename
})

table.write_parquet(
    vdb_path+vdb_filename
)

del output; del embeddings;

Loading weights:   0%|          | 0/1376 [00:00<?, ?it/s]

In [5]:
# search in base
query1 = "Мне нужна узнать: что такое кража?"
query2 = "В какой статье есть наказание за убийство?"
query3 = "Какой кодекс в арбитражном суде?"

In [6]:
ids = tokenizer(
    query2, 
    attention_mask=True,
    return_tensors="pt"
).to(device)

attention_mask = ids['attention_mask']
output = (model(**ids).last_hidden_state * attention_mask.unsqueeze(-1)).sum(1) / attention_mask.sum(1, keepdim=True)
output = F.normalize(output, dim=1)
output = output.to(torch.float64).cpu()

In [7]:
def cosine_similarity(x, y):
    return x @ y

db = pl.read_parquet(vdb_path+vdb_filename)
vector_db = torch.tensor(db['embedding_vector'].to_list(), dtype=torch.float64)
top_k_scores, top_k_indices = torch.topk(vector_db @ output.squeeze(0), k=40)

result = (
    db[top_k_indices.cpu().numpy()]
)

In [8]:
# rerank
from sentence_transformers import CrossEncoder

reranker_name = "Qwen/Qwen3-Reranker-0.6B"
reranking_model = CrossEncoder(reranker_name, cache_folder=cache_dir, device=device)

pairs = [(query2, doc) for doc in result['text']]
scores = reranking_model.predict(pairs)

del reranking_model; del model; del output

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

Default prompt name is set to 'query'. This prompt will be applied to all inference calls, except if a `prompt` or `prompt_name` parameter is provided.


In [9]:
k=5
indexes = np.argsort(scores, descending=True)[:k]
context_top_k = result[indexes]['text'].to_list()
context_top_k

['наказывается принудительными работами на срок до пяти лет либо лишением свободы на тот же срок.\nСтатья 108. Убийство, совершенное при превышении пределов необходимой обороны либо при превышении мер, необходимых для задержания лица, совершившего преступление\n1. Убийство, совершенное при превышении пределов необходимой обороны, -\nнаказывается исправительными работами на срок до двух лет, либо ограничением свободы на срок до двух лет, либо принудительными работами на срок до двух лет, либо лишением свободы на тот же срок.\n2. Убийство, совершенное при превышении мер, необходимых для задержания лица, совершившего преступление, -\nнаказывается ограничением свободы на срок до трех лет, либо принудительными работами на срок до трех лет, либо лишением свободы на тот же срок.',
 'Статья 105. Убийство\n1. Убийство, то есть умышленное причинение смерти другому человеку, -\nнаказывается лишением свободы на срок от шести до пятнадцати лет с ограничением свободы на срок до двух лет либо без так

In [10]:
from transformers import AutoModelForCausalLM

# Generate answer
k=5
indexes = np.argsort(scores, descending=True)[:k]
context_top_k = result[indexes]['text'].to_list()
context = '---'.join(context_top_k)

prompt = f"""
Ты отвечаешь на вопрос только по переданному контексту.

Правила:
- Используй только факты, прямо указанные в контексте
- Не придумывай номера статей, наказания, определения, примеры и обстоятельства
- Не объединяй сведения из разных статей, если контекст прямо не связывает их
- Если информации недостаточно, напиши: «В переданном контексте нет достаточной информации для ответа».
- Не давай юридических советов и не утверждай то, чего нет в контексте
- Ответ должен быть кратким, точным и законченным
- Ответ не должен превышать 300 слов
- Обязательно закончи все разделы в пределах этого ограничения
- Не начинай новый пункт, если не можешь закончить его в лимите

Структура ответа:

## Краткий ответ
...

## Подтверждение из контекста
- **[номер статьи или заголовок, если он есть в контексте]**: что именно сказано.
- **[номер статьи или заголовок, если он есть в контексте]**: что именно сказано.

## Ограничения
Что нельзя установить только по переданным фрагментам.

Контекст:
---
{context}
---

Вопрос: {query2}

Ответ:
"""

generative_model_name = "Qwen/Qwen3-4B-Instruct-2507"

gen_tokenizer = AutoTokenizer.from_pretrained(
    generative_model_name,
    cache_dir=cache_dir,
)

gen_model = AutoModelForCausalLM.from_pretrained(
    generative_model_name,
    cache_dir=cache_dir
).to(device).eval()
gen_model.requires_grad_(False)

messages = [{"role": "user", "content": prompt}]

inputs = gen_tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
).to(device)

with torch.inference_mode():
    generated = gen_model.generate(
        **inputs,
        max_new_tokens=2048,
    )

answer = gen_tokenizer.decode(
    generated[0][inputs["input_ids"].shape[1]:],
    skip_special_tokens=True,
)

print(answer)

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

## Краткий ответ  
В статье 105 есть наказание за убийство.

## Подтверждение из контекста  
- **Статья 105. Убийство**: 1. Убийство, то есть умышленное причинение смерти другому человеку, — наказывается лишением свободы на срок от шести до пятнадцати лет с ограничением свободы на срок до двух лет либо без такового.  
- **Статья 108. Убийство, совершенное при превышении пределов необходимой обороны либо при превышении мер, необходимых для задержания лица, совершившего преступление**: 1 и 2 части содержат наказания за убийство в определённых обстоятельствах, что также относится к убийству.

## Ограничения  
Не можно установить, в какой именно части статьи 105 или 108 указано наказание за конкретное вид убийства (например, по мотиву кровной мести), так как в контексте приведены только обобщённые формулировки без расчёта по каждому подпункту.
